# Dimensional — Seller Dimension (SCD Type 1)

Build `globalmart.gold.dim_seller` from `silver.sellers` with hash surrogate keys and MERGE upserts.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.seller_dim as seller_dim_module
importlib.reload(seller_dim_module)

from src.dimensional.seller_dim import (
    SellerDimConfig,
    build_seller_dimension_source,
    apply_seller_dimension_merge,
    run_seller_dimension,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SellerDimConfig()
print("Loaded from:", seller_dim_module.__file__)

In [ ]:
source = build_seller_dimension_source(spark, config)
apply_seller_dimension_merge(spark, source, config)
print("Sellers:", spark.table(config.target_table).count())
display(spark.table(config.target_table).limit(10))

In [ ]:
import json

report = run_seller_dimension(spark, config)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_seller.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)